# Feature Engineering – Fraud Detection (E-commerce)

## Objective
This notebook focuses on creating meaningful features from the raw e-commerce fraud dataset.
Feature engineering helps the model better understand user behavior, time patterns, and fraud signals.

We will:
- Create time-based features
- Create transaction frequency features
- Integrate geolocation (country)
- Encode categorical variables
- Scale numerical features
- Save the final processed dataset for modeling


In [5]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler


In [6]:
# Load raw datasets
fraud_df = pd.read_csv("../data/raw/Fraud_Data.csv")
ip_df = pd.read_csv("../data/raw/IpAddress_to_Country.csv")


## Step 1: Basic Data Cleaning

We remove:
- Missing values
- Duplicate transactions

This ensures data quality before feature creation.


In [7]:
fraud_df = fraud_df.dropna()
fraud_df = fraud_df.drop_duplicates()


## Step 2: Correct Data Types

- Convert signup and purchase times to datetime
- Convert IP addresses to integers for geolocation mapping


In [8]:
fraud_df["signup_time"] = pd.to_datetime(fraud_df["signup_time"])
fraud_df["purchase_time"] = pd.to_datetime(fraud_df["purchase_time"])
fraud_df["ip_address"] = fraud_df["ip_address"].astype(int)

ip_df["lower_bound_ip_address"] = ip_df["lower_bound_ip_address"].astype(int)
ip_df["upper_bound_ip_address"] = ip_df["upper_bound_ip_address"].astype(int)


## Step 3: Time-Based Feature Engineering

Fraud often happens:
- Late at night
- Shortly after signup
- On specific days

We create:
- hour_of_day
- day_of_week
- time_since_signup


In [9]:
fraud_df["hour_of_day"] = fraud_df["purchase_time"].dt.hour
fraud_df["day_of_week"] = fraud_df["purchase_time"].dt.dayofweek

fraud_df["time_since_signup"] = (
    fraud_df["purchase_time"] - fraud_df["signup_time"]
).dt.total_seconds()


## Step 4: Transaction Frequency & Velocity

Fraudsters often:
- Make many transactions quickly
- Reuse the same account

We calculate:
- Total transactions per user


In [10]:
fraud_df["transactions_per_user"] = fraud_df.groupby("user_id")["user_id"].transform("count")


## Step 5: Geolocation Feature (Country)

We map IP addresses to countries using range-based matching.
This helps detect suspicious geographic patterns.


In [11]:
ip_df = ip_df.sort_values("lower_bound_ip_address")
fraud_df = fraud_df.sort_values("ip_address")

fraud_df["country"] = pd.merge_asof(
    fraud_df,
    ip_df,
    left_on="ip_address",
    right_on="lower_bound_ip_address",
    direction="backward"
)["country"]


## Step 6: Encode Categorical Features

Machine learning models require numeric inputs.
We use One-Hot Encoding for:
- source
- browser
- sex
- country


In [12]:
fraud_df_encoded = pd.get_dummies(
    fraud_df,
    columns=["source", "browser", "sex", "country"],
    drop_first=True
)


## Step 7: Feature Scaling

We scale numerical features so no variable dominates the model.


In [13]:
num_cols = [
    "purchase_value",
    "age",
    "time_since_signup",
    "transactions_per_user"
]

scaler = StandardScaler()
fraud_df_encoded[num_cols] = scaler.fit_transform(fraud_df_encoded[num_cols])


## Step 8: Save Processed Dataset

This dataset will be used in the modeling notebook.


In [14]:
fraud_df_encoded.to_csv("../data/processed/fraud_processed.csv", index=False)
